# NB05 — GitHub Actions CI/CD Pipeline

**Module 16 — Research Software Engineering**

---

## Motivation

"It works on my machine" is not a scientific result. A CI/CD (Continuous Integration / Continuous Deployment) pipeline runs your tests automatically on every commit and pull request, on a clean machine, with the exact Python versions you specify. If tests pass on your laptop but fail in CI, you've found an environment dependency you didn't know about. If tests fail on Python 3.10 but pass on 3.12, you've found a version incompatibility before a user reports it.

## Intuition

A GitHub Actions workflow is a YAML file that tells GitHub: "whenever X happens, run these steps on a virtual machine." The most common trigger is a push or pull request. The steps are: check out the code, set up Python, install dependencies, run tests. If any step fails, the whole job fails, and GitHub marks the commit with a red X.

## Biological Background

Bioinformatics software is used on many platforms (macOS laptops, Linux HPC clusters, Docker containers) and with many Python/dependency versions. Major tools like GATK, Snakemake, and Biopython all use CI to catch cross-platform regressions. The EMBL-EBI and Broad Institute publish their CI configurations publicly — they are worth reading once you have this notebook's concepts internalized.

---

In [ ]:
import textwrap
import yaml
from pathlib import Path

# Build the CI workflow as a Python dict, then serialize to YAML
# This makes it easy to understand the structure before reading raw YAML

ci_workflow = {
    'name': 'CI',
    'on': {
        'push': {'branches': ['main', 'develop']},
        'pull_request': {'branches': ['main']},
    },
    'jobs': {
        'test': {
            'name': 'Test (Python ${{ matrix.python-version }})',
            'runs-on': 'ubuntu-latest',
            'strategy': {
                'fail-fast': False,
                'matrix': {
                    'python-version': ['3.10', '3.11', '3.12'],
                },
            },
            'steps': [
                {
                    'name': 'Check out repository',
                    'uses': 'actions/checkout@v4',
                },
                {
                    'name': 'Set up Python ${{ matrix.python-version }}',
                    'uses': 'actions/setup-python@v5',
                    'with': {'python-version': '${{ matrix.python-version }}'},
                },
                {
                    'name': 'Install dependencies',
                    'run': 'pip install -e ".[dev]"\npip install pytest pytest-cov\n',
                },
                {
                    'name': 'Run tests with coverage',
                    'run': 'pytest --cov=compbio_utils --cov-report=xml --cov-fail-under=80',
                },
                {
                    'name': 'Upload coverage report',
                    'uses': 'codecov/codecov-action@v4',
                    'with': {'file': './coverage.xml', 'fail_ci_if_error': True},
                },
            ],
        },
        'lint': {
            'name': 'Lint and type-check',
            'runs-on': 'ubuntu-latest',
            'steps': [
                {'name': 'Check out repository', 'uses': 'actions/checkout@v4'},
                {
                    'name': 'Set up Python',
                    'uses': 'actions/setup-python@v5',
                    'with': {'python-version': '3.12'},
                },
                {
                    'name': 'Install lint tools',
                    'run': 'pip install ruff mypy',
                },
                {
                    'name': 'Run ruff',
                    'run': 'ruff check src/ tests/',
                },
                {
                    'name': 'Run mypy',
                    'run': 'mypy src/compbio_utils/ --strict',
                },
            ],
        },
    },
}

yaml_content = yaml.dump(ci_workflow, default_flow_style=False, sort_keys=False)
print('Generated CI workflow YAML:')
print(yaml_content)

In [ ]:
# Write the workflow file
import tempfile
tmpdir = Path(tempfile.mkdtemp(prefix='github_actions_demo_'))
workflow_dir = tmpdir / '.github' / 'workflows'
workflow_dir.mkdir(parents=True)

workflow_path = workflow_dir / 'ci.yml'
workflow_path.write_text(yaml_content)

print(f'Wrote workflow to: {workflow_path}')
print(f'File size: {workflow_path.stat().st_size} bytes')

# Verify it round-trips correctly
reloaded = yaml.safe_load(workflow_path.read_text())
assert reloaded['name'] == 'CI'
assert '3.10' in reloaded['jobs']['test']['strategy']['matrix']['python-version']
print('Round-trip verification: PASS')

In [ ]:
# Anatomy: explain each key section
anatomy = {
    'name': 'Display name shown in the GitHub Actions tab',
    'on': 'Trigger events: push, pull_request, schedule, workflow_dispatch',
    'jobs': 'Named jobs that run in parallel by default (test, lint here)',
    'runs-on': 'OS of the virtual machine (ubuntu-latest = Ubuntu 22.04)',
    'strategy.matrix': 'Cartesian product of values — runs one job per combination',
    'steps': 'Sequential steps in a job: uses (action) or run (shell command)',
    'uses: actions/checkout@v4': 'Official action: clones the repository into the VM',
    'uses: actions/setup-python@v5': 'Official action: installs the specified Python version',
    '--cov-fail-under=80': 'Fail the job if coverage drops below 80%',
    'fail-fast: false': 'Continue running all matrix jobs even if one fails',
}

print('CI Workflow Anatomy:')
for key, explanation in anatomy.items():
    print(f'  {key:<30} {explanation}')

## Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
fig.suptitle('GitHub Actions CI Pipeline', fontsize=14, fontweight='bold')

# --- Panel 1: Pipeline flow diagram ---
ax1 = axes[0]
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 12)
ax1.axis('off')
ax1.set_title('CI Pipeline Flow', fontweight='bold')

steps = [
    (5, 11, 'git push / PR', '#2c3e50', 'TRIGGER'),
    (5, 9.2, 'actions/checkout@v4', '#3498db', 'STEP 1'),
    (5, 7.4, 'actions/setup-python@v5\nPython 3.10 | 3.11 | 3.12', '#9b59b6', 'STEP 2'),
    (5, 5.6, 'pip install -e ".[dev]"', '#e67e22', 'STEP 3'),
    (5, 3.8, 'pytest --cov --cov-fail-under=80', '#e74c3c', 'STEP 4'),
    (5, 2.0, 'codecov upload', '#27ae60', 'STEP 5'),
]

for x, y, label, color, step_num in steps:
    ax1.text(0.5, y, step_num, ha='left', va='center', fontsize=8,
             color=color, fontweight='bold')
    bbox = mpatches.FancyBboxPatch((1.5, y - 0.5), 7, 1.0,
                                    boxstyle='round,pad=0.1',
                                    facecolor=color, alpha=0.85, edgecolor='white')
    ax1.add_patch(bbox)
    ax1.text(x, y, label, ha='center', va='center', fontsize=8.5,
             color='white', fontweight='bold')

for i in range(len(steps) - 1):
    _, y1, _, _, _ = steps[i]
    _, y2, _, _, _ = steps[i + 1]
    ax1.annotate('', xy=(5, y2 + 0.5), xytext=(5, y1 - 0.5),
                 arrowprops=dict(arrowstyle='->', color='#555', lw=2))

# --- Panel 2: Pass/fail matrix across Python versions ---
ax2 = axes[1]
ax2.set_title('Test Matrix: Pass/Fail by Python Version', fontweight='bold')

python_versions = ['3.10', '3.11', '3.12']
checks = ['test_reverse_complement', 'test_gc_content', 'test_cli_gc',
          'test_cli_revcomp', 'lint (ruff)', 'type-check (mypy)']

# Simulate a matrix where all pass on 3.11+, lint fails on 3.10 (hypothetical)
results = np.ones((len(checks), len(python_versions)))
# Hypothetical: one check "fails" on 3.10 to illustrate the matrix
results[5, 0] = 0.5  # mypy warning on 3.10

im = ax2.imshow(results, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax2.set_xticks(range(len(python_versions)))
ax2.set_xticklabels([f'Python {v}' for v in python_versions])
ax2.set_yticks(range(len(checks)))
ax2.set_yticklabels(checks, fontsize=9)

status_map = {1.0: 'PASS', 0.5: 'WARN', 0.0: 'FAIL'}
color_map = {1.0: 'white', 0.5: 'black', 0.0: 'white'}

for i in range(len(checks)):
    for j in range(len(python_versions)):
        v = results[i, j]
        ax2.text(j, i, status_map.get(v, '?'),
                 ha='center', va='center', fontsize=9,
                 fontweight='bold', color=color_map.get(v, 'black'))

plt.colorbar(im, ax=ax2, label='0=FAIL  0.5=WARN  1=PASS')

plt.tight_layout()
plt.savefig('ci_pipeline.png', dpi=120, bbox_inches='tight')
plt.show()

## Exercises

See `exercises/README.md` → E05.

1. Write a GitHub Actions workflow that runs your test suite on Python 3.10 and 3.12, uploads coverage, and fails if coverage drops below 80%.
2. What does `fail-fast: false` do? When would you set it to `true`?
3. What is the difference between a CI job and a CI step?

## Quiz

1. Where do GitHub Actions workflow files live in the repository?
2. What is a matrix strategy? Give one example of when you would use it.
3. What is the difference between `uses:` and `run:` in a step?
4. What does `--cov-fail-under=80` do?
5. Why is `actions/checkout` necessary as the first step?

## Reflection

*After completing:* What is the minimum CI workflow that is actually useful? What would you cut if you could only have three steps?

## References

- GitHub Actions docs: https://docs.github.com/en/actions
- Taschuk & Wilson (2017) Rule 3: "Use a build system."